# Paper demo: sparse reconstruction

The paper's Figure 7 measures how well the largest ICA coordinates reconstruct held-out residual-stream activations. This lightweight demo applies the same top-k cosine calculation to the exact financial-context sentence used in Figure 24. It demonstrates the metric; it is not a replacement for the paper's corpus-level benchmark.

In [ ]:
%pip install -q icalens==0.4.0 matplotlib

In [ ]:
from icalens import ICALens

lens = ICALens.from_pretrained(
    "sida/icalens-gpt2-small-pile10k",
)
text = (
    "Maya stopped at the bank before the trip, waiting in line to deposit "
    "a check and withdraw enough cash for the weekend."
)
result = lens.analyze(text, layer=6)
result

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

ks = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 768]
mean_cosines = []
for k in ks:
    top_scores = lens.keep_topk(result.scores, k=k)
    reconstruction = lens.inverse_transform(top_scores, layer=6)
    similarity = F.cosine_similarity(result.activations, reconstruction, dim=-1)
    mean_cosines.append(similarity.mean().item())

plt.figure(figsize=(7, 4))
plt.plot(ks, mean_cosines, marker="o", color="#136f63")
plt.xscale("log", base=2)
plt.ylim(0, 1.02)
plt.xlabel("Retained ICA coordinates per token (k)")
plt.ylabel("Mean reconstruction cosine")
plt.title("GPT-2 Small, layer 6: paper example sentence")
plt.grid(alpha=0.25)
plt.show()

The component set is selected independently at each token by absolute signed score, matching the paper's top-k ICA reconstruction protocol. The full benchmark uses held-out Pile and FineWeb activation datasets across all model layers and compares ICA with matched SAE, PCA, and random baselines.